In [ ]:
import Augmentor as aug
import cv2
import os
from PIL import Image
import numpy as np
from split_image import split_image
from split_image import reverse_split
import rioxarray
import xarray as xr

## Various image augmentation and processing functions for use

In [ ]:
#COPIED (in part) FROM GEMINI; IMAGES MUST BE PNGs (I turned it into a looping function manually)

img_dir = fr'D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Full_Images_Masks_and_Sections_WV02_20190820222750\Sections_Color_Total'
output_dir = fr'D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Full_Images_Masks_and_Sections_WV02_20190820222750\Sections_Color_Total_PNG'

#Turns .tif files into .png files, granted their first 3 bands are RGB
def tif_to_png_mask_and_rgb(img_dir, output_dir):
    for img in os.listdir(img_dir):
        output_path = fr'{output_dir}\{img[:-4]}.png' #CHANGE THIS ACCORDINGLY
        img_work = cv2.imread(fr'{img_dir}\{img}', cv2.IMREAD_UNCHANGED) # IMREAD_UNCHANGED ensures 16-bit depths or alpha masks are kept intact
        if img is None:
            print("Error: OpenCV still cannot read this file structure or the path is broken.")
        else:
            cv2.imwrite(output_path, img_work)
            print("Conversion successful!")

#Turns .tif files into .png files, granted their only band is NDVI
def tif_to_png_ndvi(img_dir, output_dir):
    for img in os.listdir(img_dir):
        output_path = fr'{output_dir}\{img[:-4]}.png'
        img_work = cv2.imread(fr'{img_dir}\{img}', cv2.IMREAD_UNCHANGED) # IMREAD_UNCHANGED ensures 16-bit depths or alpha masks are kept intact
        img_work = (((img_work)+1)/2)*255 #Normalizes it; starts by converting range of -1 to 1 to 0 to 2, then divides it by 2 to create range from 0 to 1; a ratio that can be multiplied to 255, the band scale of a PNG
        if img is None:
            print("Error: OpenCV still cannot read this file structure or the path is broken.")
        else:
            cv2.imwrite(output_path, img_work)
            print("Conversion successful!")

#Turns .tif files into .png files, granted their only band is slope
def tif_to_png_slope(img_dir, output_dir):
    for img in os.listdir(img_dir):
        output_path = fr'{output_dir}\{img[:-4]}.png'
        img_work = cv2.imread(fr'{img_dir}\{img}', cv2.IMREAD_UNCHANGED) # IMREAD_UNCHANGED ensures 16-bit depths or alpha masks are kept intact
        img_work = (((np.maximum(img_work,0)))/90)*255 #Normalizes it; starts by converting any anomalous negative values to zero, then normalizes from 0 to 1 based on range from 0 to 90, then multiplies with 255 for png grayscale
        if img is None:
            print("Error: OpenCV still cannot read this file structure or the path is broken.")
        else:
            cv2.imwrite(output_path, img_work)
            print("Conversion successful!")

#Turns RGBA images into RGB images
def rgba_to_rgb(rgba_directory, rgb_directory):
    for item in os.listdir(rgba_directory):
        img = Image.open(fr'{rgba_directory}/{item}')
        img_rgb = img.convert('RGB')
        img_rgb.save(fr'{rgb_directory}/{item}')

#Runs an augmentation script to create augmented training images
def augmentation_script(image_path, mask_path):
    p = aug.Pipeline(image_path)
    p.ground_truth(mask_path)
    p.rotate90(probability = 0.33)
    p.rotate270(probability = 0.33)
    p.flip_left_right(probability = 0.7)
    p.flip_top_bottom(probability = 0.4)
    p.crop_random(probability=1, percentage_area=0.25)
    p.resize(probability=1.0, width=256, height=256)
    p.sample(200)

#Runs a script to divide images into 16 pieces (4,4 are rows and columns)
def section_divider(input_path, output_path):
    split_image(input_path, 4, 4, False, False, output_dir=output_path)


#This function sorts the sets of 16 images from section_divider into a dictionary
def section_creator(input_path, section_sum = 16):
    start = 0
    end = section_sum
    d = 0
    subsections = {}
    directory_list = sorted(os.listdir(input_path))
    
    while end <= len(directory_list):
        subsections[d] = directory_list[start:end]
        d += 1
        start = end
        end = end + section_sum
    return subsections

#Recombines the images from section_divider using the section_creator script
def section_recombination_four_by_four(input_path, output_path, section_sum = 16):
    sections = section_creator(input_path, section_sum)
    for item in range(0,len(sections)):
        img_1_1 = cv2.imread(fr'{input_path}/{sections[item][0]}')
        img_1_2 = cv2.imread(fr'{input_path}/{sections[item][7]}')
        img_1_3 = cv2.imread(fr'{input_path}/{sections[item][8]}')
        img_1_4 = cv2.imread(fr'{input_path}/{sections[item][9]}')
        
        img_2_1 = cv2.imread(fr'{input_path}/{sections[item][10]}')
        img_2_2 = cv2.imread(fr'{input_path}/{sections[item][11]}')
        img_2_3 = cv2.imread(fr'{input_path}/{sections[item][12]}')
        img_2_4 = cv2.imread(fr'{input_path}/{sections[item][13]}')

        img_3_1 = cv2.imread(fr'{input_path}/{sections[item][14]}')
        img_3_2 = cv2.imread(fr'{input_path}/{sections[item][15]}')
        img_3_3 = cv2.imread(fr'{input_path}/{sections[item][1]}')
        img_3_4 = cv2.imread(fr'{input_path}/{sections[item][2]}')

        img_4_1 = cv2.imread(fr'{input_path}/{sections[item][3]}')
        img_4_2 = cv2.imread(fr'{input_path}/{sections[item][4]}')
        img_4_3 = cv2.imread(fr'{input_path}/{sections[item][5]}')
        img_4_4 = cv2.imread(fr'{input_path}/{sections[item][6]}')

        h_1 = np.hstack([img_1_1, img_1_2, img_1_3, img_1_4])
        h_2 = np.hstack([img_2_1, img_2_2, img_2_3, img_2_4])
        h_3 = np.hstack([img_3_1, img_3_2, img_3_3, img_3_4])
        h_4 = np.hstack([img_4_1, img_4_2, img_4_3, img_4_4])

        complete = np.vstack([h_1, h_2, h_3, h_4])

        cv2.imwrite(fr'{output_path}/{sections[item][0][-19:-4]}.png', complete)


#Gemini and stackexchange assisted
def multispectral_to_rgb_png(input_dir, rgb_dir, rgb_bands = [1,2,3]):
    # -------------------------------------------------------------------------------------------
    # --- OLD CODE FOR SAMPLING FOR RGB; CLOSE TO CORRECT BUT DOES NOT MIMIC QGIS ---------------
    # -------------------------------------------------------------------------------------------
    # true_red_max = 0
    # true_red_min = float('inf')
    # true_blue_max = 0
    # true_blue_min = float('inf')
    # true_green_max = 0
    # true_green_min = float('inf')

    # for item in os.listdir(input_dir):
    #     img = rioxarray.open_rasterio(fr'{input_dir}/{item}')
    #     img_ds = img.to_dataset(dim='band')

    #     red = img_ds[rgb_bands[0]].to_numpy()
    #     red_max, red_min = np.percentile(red, 98), np.percentile(red[red != 0], 2)
    #     green = img_ds[rgb_bands[1]].to_numpy()
    #     green_max, green_min = np.percentile(green, 98), np.percentile(green[green != 0], 2)
    #     blue = img_ds[rgb_bands[2]].to_numpy()
    #     blue_max, blue_min = np.percentile(blue, 98), np.percentile(blue[blue != 0], 2)

    #     if red_max > true_red_max:
    #         true_red_max = red_max
    #     if blue_max > true_blue_max:
    #         true_blue_max = blue_max
    #     if green_max > true_green_max:
    #         true_green_max = green_max

    #     if red_min < true_red_min:
    #         true_red_min = red_min
    #     if blue_min < true_blue_min:
    #         true_blue_min = blue_min
    #     if green_min < true_green_min:
    #         true_green_min = green_min
    # -------------------------------------------------------------------------------------------
    # -------------------------------------------------------------------------------------------
    # -------------------------------------------------------------------------------------------


    # #-------------------------------------------------------------------------------------------
    # #--- GEMINI COPIED CODE (mostly); THIS CODE MIMICS THE WAY QGIS SAMPLES IMAGES TO CREATE RGB VIEW ---
    # #-------------------------------------------------------------------------------------------
    red_samples, green_samples, blue_samples = [], [], []

    for item in os.listdir(input_dir):
        if not item.endswith(('.tif', '.tiff')):
            continue
        img = rioxarray.open_rasterio(f"{input_dir}/{item}")
        img_ds = img.to_dataset(dim='band')
        img_ds[rgb_bands[0]] = xr.where(img_ds[rgb_bands[0]]==65535.0, 0, img_ds[rgb_bands[0]])
        img_ds[rgb_bands[1]] = xr.where(img_ds[rgb_bands[1]]==65535.0, 0, img_ds[rgb_bands[1]])
        img_ds[rgb_bands[2]] = xr.where(img_ds[rgb_bands[2]]==65535.0, 0, img_ds[rgb_bands[2]])
        
        r = img_ds[rgb_bands[0]].to_numpy().flatten()
        g = img_ds[rgb_bands[1]].to_numpy().flatten()
        b = img_ds[rgb_bands[2]].to_numpy().flatten()
        
        # QGIS ignores 0/NoData when computing statistics. 
        # We sample every 10th pixel to avoid crashing system memory on large areas.
        red_samples.extend(r[(r > 0)][::10])
        green_samples.extend(g[(g > 0)][::10])
        blue_samples.extend(b[(b > 0)][::10])

    # Calculate actual cumulative count cuts (2% - 98%) globally across the full image space
    true_red_min, true_red_max = np.percentile(red_samples, 2), np.percentile(red_samples, 98)
    true_green_min, true_green_max = np.percentile(green_samples, 2), np.percentile(green_samples, 98)
    true_blue_min, true_blue_max = np.percentile(blue_samples, 2), np.percentile(blue_samples, 98)
    # #-------------------------------------------------------------------------------------------
    # #-------------------------------------------------------------------------------------------
    # #-------------------------------------------------------------------------------------------



    for item in os.listdir(input_dir):
        img = rioxarray.open_rasterio(fr'{input_dir}/{item}')
        img_ds = img.to_dataset(dim='band')

        red = img_ds[rgb_bands[0]].to_numpy()
        normalized_red = np.where(red != 0, ((red-true_red_min)/(true_red_max-true_red_min))*255, 0)
        normalized_red = np.clip(normalized_red, 0, 255).astype(np.uint8)

        green = img_ds[rgb_bands[1]].to_numpy()
        normalized_green = np.where(green != 0, ((green-true_green_min)/(true_green_max-true_green_min))*255, 0)
        normalized_green = np.clip(normalized_green, 0, 255).astype(np.uint8)


        blue = img_ds[rgb_bands[2]].to_numpy()
        normalized_blue = np.where(blue != 0, ((blue-true_blue_min)/(true_blue_max-true_blue_min))*255, 0)
        normalized_blue = np.clip(normalized_blue, 0, 255).astype(np.uint8)

        rgb = np.dstack((normalized_red, normalized_green, normalized_blue))

        rgb_image = Image.fromarray(rgb)
        rgb_image.save(fr'{rgb_dir}/{item[:-4]}.png')

large_dir = fr"D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Full_Images_Masks_and_Sections_WV02_20190820222750\Sections_Color_Total_PNG"
filtered_dir = fr"D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Full_Images_Masks_and_Sections_WV02_20190820222750\Sections_Color_Total_PNG_Filtered"
section_filtered_dir = fr"D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Full_Images_Masks_and_Sections_WV02_20190820222750\Sections_Color_Total_PNG_Filtered_256"

def filterer_and_divider(large_dir, filtered_dir, section_filtered_dir, size=1024, bands=3):
    #Filters out images that are not square in dimension and have all 0 values for the first band
    for item in os.listdir(large_dir):
        img = cv2.imread(fr'{large_dir}/{item}')
        if img.shape[0] == img.shape[1]:
            unique, counts = np.unique(img, return_counts=True)
            if counts[0] != (size*size*bands):
                cv2.imwrite(fr'{filtered_dir}/{item}', img)

    #Divides these selected images into 16ths
    for item in os.listdir(filtered_dir):
        section_divider(input_path=fr'{large_dir}/{item}', output_path=section_filtered_dir)

def overlay_mask_on_image(image, mask, alpha=0.5):
    # Ensure both are the same size
    if image.shape[:2] != mask.shape[:2]:
        raise ValueError("Image and mask must be the same size")

    # Convert to float32 for accurate blending
    image = image.astype(np.float32)
    mask = mask.astype(np.float32)

    # Blend the mask and the image
    beta = 1 - alpha
    gamma = 1
    overlay = cv2.addWeighted(mask, alpha, image, beta, gamma)
    overlayed = overlay.astype(np.uint8)

    # Convert back to BGR to save with OpenCV
    overlayed_bgr = cv2.cvtColor(overlayed, cv2.COLOR_RGB2BGR)
    cv2.imwrite(fr"/home/gaimholte/Imholte_segmentation_gym_ubuntu/segmentation_gym-main/High_Res_WT_Data/WT_Dataset_3B/WT_Data/toPredict/overlayed_result.png", overlayed_bgr)


image = cv2.imread(fr"/home/gaimholte/Imholte_segmentation_gym_ubuntu/segmentation_gym-main/High_Res_WT_Data/WT_Dataset_3B/WT_Data/toPredict/Adjusted_WV02_20190820222750_10300100968FB300_19AUG20222750-M1BS-503550221040_01_P004_u16rf3413_06_09.png", 
                   cv2.IMREAD_COLOR)
mask = cv2.imread(fr"/home/gaimholte/Imholte_segmentation_gym_ubuntu/segmentation_gym-main/High_Res_WT_Data/WT_Dataset_3B/WT_Data/toPredict/full/Adjusted_WV02_20190820222750_10300100968FB300_19AUG20222750-M1BS-503550221040_01_P004_u16rf3413_06_09_predseg.png",
                   cv2.IMREAD_COLOR)

overlay_mask_on_image(image,mask)

# **Workflow for image processing, starting with a Worldview-2 .tif**

In [45]:
input_tif_tile_dir = fr"D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Images_WV02_20180726231451\Sections_Total"
output_png_dir = fr"D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Images_WV02_20180726231451\Sections_Total_PNG"
output_filtered_dir = fr"D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Images_WV02_20180726231451\Sections_Total_Filtered_PNG"
output_filtered_section_dir = fr"D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Images_WV02_20180726231451\Sections_Total_Filtered_PNG_256"

multispectral_to_rgb_png(input_dir=input_tif_tile_dir, rgb_dir=output_png_dir, rgb_bands=[5,3,2])
filterer_and_divider(large_dir=output_png_dir, filtered_dir=output_filtered_dir, section_filtered_dir=output_filtered_section_dir)

Exporting image tile: D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Images_WV02_20180726231451\Sections_Total_Filtered_PNG_256\WV02_20180726231451_1030010081A41700_18JUL26231451-M1BS-502528971070_01_P003_u16rf3413_01_04_0.png
Exporting image tile: D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Images_WV02_20180726231451\Sections_Total_Filtered_PNG_256\WV02_20180726231451_1030010081A41700_18JUL26231451-M1BS-502528971070_01_P003_u16rf3413_01_04_1.png
Exporting image tile: D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Images_WV02_20180726231451\Sections_Total_Filtered_PNG_256\WV02_20180726231451_1030010081A41700_18JUL26231451-M1BS-502528971070_01_P003_u16rf3413_01_04_2.png
Exporting image tile: D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Images_WV02_20180726231451\Sections_Total_Filtered_PNG_256\WV02_20180726231451_1030010081A41700_18JUL26231451-M1BS-502528971070_01_P003_u16rf3413_01_04_3.png
Exporting image tile: D:\Imholte_Research_WT

## *Now, move these tiles into WSL Ubuntu and run seg_images_in_folder.py on them*

In [46]:
input_small_dir = fr"D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Images_WV02_20180726231451\Sections_ML_Mask_Test\256_from_Model"
output_large_dir = fr"D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Images_WV02_20180726231451\Sections_ML_Mask_Test\1024_from_Model"

section_recombination_four_by_four(input_path=input_small_dir, output_path=output_large_dir)

## *Lastly, run the georeferencing mask script in its notebook*